In [4]:
from __future__ import annotations
from pathlib import Path
import random
import pandas as pd

def resolveDatasetsRoot() -> Path:
    candidates = [
        Path.cwd() / 'notebooks' / 'datasets',
        Path.cwd() / 'datasets',
        Path.cwd().parent / 'notebooks' / 'datasets',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return Path('notebooks/datasets')

DATASETS_ROOT = resolveDatasetsRoot()
OUTPUT_ROOT = Path('data/parquet')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TARGET_ROWS = 15000
CHUNK_SIZE = 50000
rng = random.Random(RANDOM_SEED)

def sampleFrame(frame: pd.DataFrame, targetRows: int, rng: random.Random) -> pd.DataFrame:
    if len(frame) <= targetRows:
        return frame
    return frame.sample(n=targetRows, random_state=rng.randint(0, 2**31 - 1)).reset_index(drop=True)

def reservoirSample(frame: pd.DataFrame, reservoir: pd.DataFrame, targetRows: int, rng: random.Random) -> pd.DataFrame:
    if reservoir.empty:
        return sampleFrame(frame, targetRows, rng)
    combined = pd.concat([reservoir, frame], ignore_index=True)
    return sampleFrame(combined, targetRows, rng)

def sampleCsv(path: Path, targetRows: int, rng: random.Random) -> pd.DataFrame:
    reservoir = pd.DataFrame()
    for chunk in pd.read_csv(path, chunksize=CHUNK_SIZE):
        reservoir = reservoirSample(chunk, reservoir, targetRows, rng)
    return reservoir

def sampleJsonl(path: Path, targetRows: int, rng: random.Random) -> pd.DataFrame:
    reservoir = pd.DataFrame()
    for chunk in pd.read_json(path, lines=True, chunksize=CHUNK_SIZE):
        reservoir = reservoirSample(chunk, reservoir, targetRows, rng)
    return reservoir

def sanitizeFrame(frame: pd.DataFrame) -> pd.DataFrame:
    cleaned = frame.copy()
    cleaned.replace({'—': None, 'N/A': None, 'n/a': None}, inplace=True)
    for column in cleaned.columns:
        if cleaned[column].dtype == 'object':
            numeric = pd.to_numeric(cleaned[column], errors='coerce')
            nonNullRatio = numeric.notna().mean()
            if nonNullRatio >= 0.8:
                cleaned[column] = numeric
            else:
                cleaned[column] = cleaned[column].astype('string')
    return cleaned

def writeParquet(frame: pd.DataFrame, name: str) -> Path:
    outputPath = OUTPUT_ROOT / f'{name}.parquet'
    safeFrame = sanitizeFrame(frame)
    safeFrame.to_parquet(outputPath, index=False)
    return outputPath

In [5]:
datasetJobs = [
    {
        'name': 'amazon_all_beauty_reviews',
        'path': DATASETS_ROOT / 'Amazon' / 'AllBeauty' / 'All_Beauty.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'amazon_all_beauty_meta',
        'path': DATASETS_ROOT / 'Amazon' / 'AllBeauty' / 'meta_All_Beauty.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'amazon_fashion_reviews',
        'path': DATASETS_ROOT / 'Amazon' / 'AmazonFashion' / 'Amazon_Fashion.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'amazon_fashion_meta',
        'path': DATASETS_ROOT / 'Amazon' / 'AmazonFashion' / 'meta_Amazon_Fashion.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'amazon_beauty_personal_care_reviews',
        'path': DATASETS_ROOT / 'Amazon' / 'BeautyandPersonalCare' / 'Beauty_and_Personal_Care.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'amazon_beauty_personal_care_meta',
        'path': DATASETS_ROOT / 'Amazon' / 'BeautyandPersonalCare' / 'meta_Beauty_and_Personal_Care.jsonl',
        'type': 'jsonl',
    },
    {
        'name': 'cosmetics_events_2019_oct',
        'path': DATASETS_ROOT / 'eCommerceEventsHistoryinCosmeticsShop' / '2019-Oct.csv',
        'type': 'csv',
    },
    {
        'name': 'cosmetics_events_2019_nov',
        'path': DATASETS_ROOT / 'eCommerceEventsHistoryinCosmeticsShop' / '2019-Nov.csv',
        'type': 'csv',
    },
    {
        'name': 'cosmetics_events_2019_dec',
        'path': DATASETS_ROOT / 'eCommerceEventsHistoryinCosmeticsShop' / '2019-Dec.csv',
        'type': 'csv',
    },
    {
        'name': 'cosmetics_events_2020_jan',
        'path': DATASETS_ROOT / 'eCommerceEventsHistoryinCosmeticsShop' / '2020-Jan.csv',
        'type': 'csv',
    },
    {
        'name': 'cosmetics_events_2020_feb',
        'path': DATASETS_ROOT / 'eCommerceEventsHistoryinCosmeticsShop' / '2020-Feb.csv',
        'type': 'csv',
    },
    {
        'name': 'global_trending_hashtags',
        'path': DATASETS_ROOT / 'GlobalTrendingHastags' / 'trending_hashtags.csv',
        'type': 'csv',
    },
    {
        'name': 'womens_clothing_reviews',
        'path': DATASETS_ROOT / 'WomensClothingE-CommerceReviews' / 'Womens Clothing E-Commerce Reviews.csv',
        'type': 'csv',
    },
]

def runSampling(jobs: list[dict[str, str]]) -> pd.DataFrame:
    results = []
    for job in jobs:
        path = Path(job['path'])
        if not path.exists():
            results.append({'name': job['name'], 'rows': 0, 'status': 'missing'})
            continue
        if job['type'] == 'csv':
            sample = sampleCsv(path, TARGET_ROWS, rng)
        else:
            sample = sampleJsonl(path, TARGET_ROWS, rng)
        outputPath = writeParquet(sample, job['name'])
        results.append({'name': job['name'], 'rows': len(sample), 'status': str(outputPath)})
    return pd.DataFrame(results)

In [6]:
results = runSampling(datasetJobs)
results

,name,rows,status
0,amazon_all_beauty_reviews,15000,data\parquet\amazon_all_beauty_reviews.parquet
1,amazon_all_beauty_meta,15000,data\parquet\amazon_all_beauty_meta.parquet
2,amazon_fashion_reviews,15000,data\parquet\amazon_fashion_reviews.parquet
3,amazon_fashion_meta,15000,data\parquet\amazon_fashion_meta.parquet
4,amazon_beauty_personal_care_reviews,15000,data\parquet\amazon_beauty_personal_care_revie...
5,amazon_beauty_personal_care_meta,15000,data\parquet\amazon_beauty_personal_care_meta....
6,cosmetics_events_2019_oct,15000,data\parquet\cosmetics_events_2019_oct.parquet
7,cosmetics_events_2019_nov,15000,data\parquet\cosmetics_events_2019_nov.parquet
8,cosmetics_events_2019_dec,15000,data\parquet\cosmetics_events_2019_dec.parquet
9,cosmetics_events_2020_jan,15000,data\parquet\cosmetics_events_2020_jan.parquet
